# GEPA Skill Optimization Pipeline

This notebook demonstrates how to optimize Claude Agent Skills using **DSPy's GEPA (Greedy Evolutionary Prompt Adaptation)** optimizer with **OpenAI GPT** models.

## Overview

1. **Setup** - Import dependencies and configure OpenAI
2. **Load Skill** - Parse skill directory
3. **Analyze** - Check against best practices
4. **Define Module** - Create DSPy module for optimization
5. **Prepare Data** - Create training examples
6. **Define Metric** - Scoring function for GEPA
7. **Run GEPA** - Evolutionary prompt optimization
8. **Apply & Save** - Use optimized module and save results

## 1. Setup

In [4]:
# Install dependencies (run once)
# !pip install -e .
# !pip install dspy-ai openai python-dotenv

In [1]:
from pathlib import Path
from skillopt import SkillParser, SkillAnalyzer
import dspy
import os
from dotenv import load_dotenv

load_dotenv()

parser = SkillParser()
analyzer = SkillAnalyzer()

print("SkillOpt components initialized")

SkillOpt components initialized


In [2]:
# Available models: gpt-4o, gpt-4o-mini, gpt-4-turbo, gpt-3.5-turbo
lm = dspy.LM(
    "openai/gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY")
)

dspy.configure(lm=lm)
print(f"Configured DSPy with: {lm.model}")

Configured DSPy with: openai/gpt-4o


## 2. Load Skill

In [3]:
# Path to your skill directory
SKILL_PATH = Path("examples/Bad_Kubernetes_Helper_Skill")

# Parse the skill
if SKILL_PATH.exists():
    skill = parser.parse_directory(SKILL_PATH)
    print(f"Loaded skill: {skill.name}")
    print(f"  Lines: {skill.total_lines}")
    print(f"  References: {len(skill.references)}")
else:
    print(f"Skill not found at: {SKILL_PATH}")
    skill = None

Loaded skill: Bad_Kubernetes_Helper_Skill
  Lines: 565
  References: 2


## 3. Analyze Skill

In [4]:
# Analyze skill against best practices
if skill:
    report = analyzer.analyze(skill)
    
    print(f"Analysis Report: {skill.name}")
    print("="*50)
    print(f"Score: {report.score}/100")
    print(f"Issues: {len(report.issues)}")
    print(f"")
    print("Issues by severity:")
    errors = [i for i in report.issues if i.severity == 'error']
    warnings = [i for i in report.issues if i.severity == 'warning']
    suggestions = [i for i in report.issues if i.severity == 'suggestion']
    print(f"  Errors: {len(errors)}")
    print(f"  Warnings: {len(warnings)}")
    print(f"  Suggestions: {len(suggestions)}")

Analysis Report: Bad_Kubernetes_Helper_Skill
Score: 0/100
Issues: 22

Issues by severity:
  Errors: 2
  Warnings: 16
  Suggestions: 4


In [5]:
# View detailed issues
if skill and 'report' in dir():
    print("Detailed Issues:")
    print("="*50)
    for issue in report.issues[:-1]:
        icon = {"error": "[ERR]", "warning": "[WRN]", "suggestion": "[SUG]"}.get(issue.severity, "[?]")
        print(f"{icon} {issue.category}: {issue.message}")

Detailed Issues:
[ERR] structure: SKILL.md has 502 lines, should be under 500
[WRN] conciseness: Filler phrase 'it is important to' appears 14 time(s)
[WRN] conciseness: Filler phrase 'please note that' appears 11 time(s)
[WRN] conciseness: Filler phrase 'keep in mind' appears 7 time(s)
[WRN] conciseness: Filler phrase 'make sure to' appears 40 time(s)
[WRN] conciseness: Filler phrase 'remember to' appears 9 time(s)
[WRN] conciseness: Filler phrase 'ensure that' appears 40 time(s)
[WRN] conciseness: Filler phrase 'you should' appears 12 time(s)
[WRN] conciseness: Filler phrase 'you can' appears 6 time(s)
[WRN] conciseness: Filler phrase 'you'll need to' appears 1 time(s)
[WRN] conciseness: Filler phrase 'first, you'll' appears 1 time(s)
[WRN] conciseness: Filler phrase 'don't forget to' appears 12 time(s)
[WRN] conciseness: Filler phrase 'you need to' appears 3 time(s)
[WRN] conciseness: Assumes Claude doesn't know Kubernetes
[WRN] structure: Reference advanced-topics.md links to other

## 4. Define DSPy Module

In [ ]:
# Define DSPy Signature and Module for Skill Optimization
import re

class OptimizeSkillContent(dspy.Signature):
    """Optimize a Claude Agent Skill following best practices.

    GOAL: The optimized skill must work exactly as the original. Reduce token usage
    while preserving full functionality. Apply training examples and best practices
    to create a well-structured, concise skill.

    PRESERVE:
    - Core functionality and workflow logic
    - Essential code blocks (```bash, ```python, etc.)
    - Frontmatter (--- name/description ---)
    - Section headers (##)
    - Unique instructions and commands

    REMOVE:
    - Filler phrases: "make sure to", "ensure that", "don't forget to", etc.
    - Verbose explanations of concepts Claude already knows (YAML, JSON, APIs)
    - Repetitive instructions and duplicate commands
    - Unrelated content (e.g., PDF/Git commands in a Kubernetes skill)
    - Time-sensitive information

    CONSOLIDATE:
    - Similar commands into one with placeholders (e.g., -n <namespace>)
    - Repetitive steps into concise statements
    """
    original_content: str = dspy.InputField(desc="Original skill content with issues")
    issues_found: str = dspy.InputField(desc="List of issues identified in the skill")
    optimized_content: str = dspy.OutputField(desc="Optimized skill preserving code blocks but removing filler")

class SkillOptimizerModule(dspy.Module):
    """DSPy module for skill optimization that preserves code blocks."""
    
    def __init__(self):
        super().__init__()
        self.optimizer = dspy.ChainOfThought(OptimizeSkillContent)
    
    def forward(self, original_content: str, issues_found: str) -> str:
        result = self.optimizer(
            original_content=original_content,
            issues_found=issues_found
        )
        return result.optimized_content

# Helper function to extract code blocks
def extract_code_blocks(content: str) -> list[str]:
    """Extract all code blocks from content."""
    pattern = r'```(?:\w+)?\s*\n(.*?)```'
    return re.findall(pattern, content, re.DOTALL)

def count_code_blocks(content: str) -> int:
    """Count number of code blocks."""
    return len(re.findall(r'```', content)) // 2

# Initialize the module
skill_optimizer_module = SkillOptimizerModule()
print("Created SkillOptimizerModule with code block preservation")

## 5. Prepare Training Data

In [7]:
# Prepare training examples for GEPA
# Format issues as string
issues_str = "\n".join([
    f"- [{issue.severity}] {issue.category}: {issue.message}"
    for issue in report.issues[:20]
])

# Use more content to capture code blocks (up to 8000 chars)
# This includes the kubectl command sections
skill_content = skill.main_file.raw_content[:8000]

# Count code blocks in the content
original_code_blocks = count_code_blocks(skill_content)

print(f"Skill content: {len(skill_content):,} chars")
print(f"Code blocks found: {original_code_blocks}")
print()

# Create training example
trainset = [
    dspy.Example(
        original_content=skill_content,
        issues_found=issues_str
    ).with_inputs("original_content", "issues_found")
]

print(f"Created {len(trainset)} training example(s)")
print()
print("Issues to fix:")
print(issues_str[:500] + "..." if len(issues_str) > 500 else issues_str)

Skill content: 8,000 chars
Code blocks found: 14

Created 1 training example(s)

Issues to fix:
- [error] structure: SKILL.md has 502 lines, should be under 500
- [warning] conciseness: Filler phrase 'it is important to' appears 14 time(s)
- [warning] conciseness: Filler phrase 'please note that' appears 11 time(s)
- [warning] conciseness: Filler phrase 'keep in mind' appears 7 time(s)
- [warning] conciseness: Filler phrase 'make sure to' appears 40 time(s)
- [warning] conciseness: Filler phrase 'remember to' appears 9 time(s)
- [warning] conciseness: Filler phrase 'ensure that' appears 40...


### Load Best Practices Training Examples

Load curated bad/good example pairs from the official Claude Agent Skills best practices documentation.

In [9]:
# Load best practices training examples from JSON
import json

TRAINING_EXAMPLES_PATH = Path("examples/training_examples.json")

def load_training_examples(json_path=TRAINING_EXAMPLES_PATH, categories=None):
    """Load training examples from JSON file."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    examples = data.get("examples", [])
    
    if categories:
        examples = [ex for ex in examples if ex.get("category") in categories]
    
    return data, examples

def examples_to_dspy_trainset(examples):
    """Convert JSON examples to DSPy format."""
    trainset = []
    
    for ex in examples:
        if "bad" not in ex or "good" not in ex:
            continue
        
        bad = ex["bad"]
        good = ex["good"]
        
        if isinstance(bad, dict):
            bad_content = bad.get("content", "")
            bad_issues = bad.get("issues", [])
        else:
            bad_content = str(bad)
            bad_issues = []
        
        if isinstance(good, dict):
            good_content = good.get("content", "")
        else:
            good_content = str(good)
        
        issues_str = "\n".join([
            f"- [{ex.get('category', 'unknown')}] {issue}"
            for issue in bad_issues
        ])
        
        if not issues_str:
            issues_str = f"- [{ex.get('category', 'unknown')}] {ex.get('principle', 'Needs optimization')}"
        
        dspy_example = dspy.Example(
            original_content=bad_content,
            issues_found=issues_str,
            optimized_content=good_content,
        ).with_inputs("original_content", "issues_found")
        
        trainset.append(dspy_example)
    
    return trainset

# Load all training data
training_data, all_examples = load_training_examples()

print(f"Loaded training examples from: {TRAINING_EXAMPLES_PATH}")
print(f"  Total examples: {len(all_examples)}")
print(f"  Categories: {training_data.get('categories', [])}")
print(f"  Filler phrases to remove: {len(training_data.get('filler_phrases_to_remove', []))}")
print(f"  Verbose patterns: {len(training_data.get('verbose_patterns_to_simplify', []))}")

Loaded training examples from: examples/training_examples.json
  Total examples: 30
  Categories: ['conciseness', 'frontmatter', 'naming', 'structure', 'content', 'scripts', 'terminology', 'workflows', 'templates', 'paths']
  Filler phrases to remove: 13
  Verbose patterns: 10


In [10]:
# Convert best practices examples to DSPy trainset
best_practices_trainset = examples_to_dspy_trainset(all_examples)

print(f"Converted {len(best_practices_trainset)} examples to DSPy format")
print()

# Show examples by category
category_counts = {}
for ex in all_examples:
    cat = ex.get("category", "unknown")
    category_counts[cat] = category_counts.get(cat, 0) + 1

print("Examples by category:")
for cat, count in sorted(category_counts.items()):
    print(f"  - {cat}: {count}")

print()
print("Sample example (conciseness - PDF extraction):")
if best_practices_trainset:
    sample = best_practices_trainset[0]
    print(f"  Original: {sample.original_content[:100]}...")
    print(f"  Issues: {sample.issues_found[:80]}...")
    print(f"  Optimized: {sample.optimized_content[:100]}...")

Converted 29 examples to DSPy format

Examples by category:
  - conciseness: 4
  - content: 1
  - frontmatter: 5
  - naming: 2
  - paths: 1
  - scripts: 4
  - structure: 6
  - templates: 3
  - terminology: 1
  - workflows: 3

Sample example (conciseness - PDF extraction):
  Original: ## Extract PDF text

PDF (Portable Document Format) files are a common file format that contains
tex...
  Issues: - [conciseness] Explains what PDFs are (Claude knows this)
- [conciseness] Expla...
  Optimized: ## Extract PDF text

Use pdfplumber for text extraction:

```python
import pdfplumber

with pdfplumb...


In [ ]:
# Combine skill-specific example with best practices examples

# Skill-specific example
trainset_skill = [
    dspy.Example(
        original_content=skill_content,
        issues_found=issues_str
    ).with_inputs("original_content", "issues_found")
]

# Combined: skill + all best practices examples (same as CLI)
trainset = trainset_skill + best_practices_trainset

print(f"Training set:")
print(f"  Skill example: 1")
print(f"  Best practices: {len(best_practices_trainset)}")
print(f"  Total: {len(trainset)} examples")

## 6. Define Metric Function

In [12]:
# Define metric function for GEPA optimization

def skill_optimization_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """Evaluate skill optimization quality.
    
    Scoring:
    - 25% Filler phrase removal
    - 20% Conciseness (but not too aggressive)
    - 25% Code block preservation (CRITICAL)
    - 15% Structure preserved (frontmatter, sections)
    - 15% Verbose explanations removed
    
    Returns:
        float: Score between 0 and 1
    """
    optimized = pred.optimized_content if hasattr(pred, 'optimized_content') else str(pred)
    original_content = gold.original_content if hasattr(gold, 'original_content') else ""
    
    score = 0.0
    
    # 1. Filler phrase removal (25%)
    filler_phrases = ['make sure to', 'ensure that', "don't forget to", 'remember to', 
                      'it is important to', 'please note that', 'keep in mind',
                      'you should', 'you need to', 'you must']
    filler_count = sum(1 for phrase in filler_phrases if phrase.lower() in optimized.lower())
    filler_score = max(0, 1 - (filler_count / 5))
    score += 0.25 * filler_score
    
    # 2. Conciseness - target 40-70% reduction, penalize over-reduction (20%)
    original_len = len(original_content) if original_content else 1
    optimized_len = len(optimized)
    if optimized_len < original_len:
        reduction_ratio = 1 - (optimized_len / original_len)
        if reduction_ratio < 0.4:
            concise_score = reduction_ratio / 0.4  # Less than 40% reduction
        elif reduction_ratio <= 0.7:
            concise_score = 1.0  # Sweet spot: 40-70% reduction
        else:
            concise_score = max(0, 1 - (reduction_ratio - 0.7) * 3)  # Penalize over-reduction
    else:
        concise_score = 0.0
    score += 0.20 * concise_score
    
    # 3. Code block preservation - CRITICAL (25%)
    original_blocks = count_code_blocks(original_content)
    optimized_blocks = count_code_blocks(optimized)
    
    if original_blocks > 0:
        # Reward keeping code blocks, penalize removing them
        block_ratio = optimized_blocks / original_blocks
        if block_ratio >= 0.5:  # Keep at least 50% of code blocks (consolidated)
            code_score = min(1.0, block_ratio)
        else:
            code_score = block_ratio * 0.5  # Heavy penalty for removing too many
    else:
        code_score = 1.0 if optimized_blocks == 0 else 0.8
    score += 0.25 * code_score
    
    # 4. Structure preserved - frontmatter and sections (15%)
    has_frontmatter = optimized.strip().startswith('---')
    has_sections = '## ' in optimized
    has_code = '```' in optimized
    structure_score = (
        (0.4 if has_frontmatter else 0) + 
        (0.3 if has_sections else 0) +
        (0.3 if has_code else 0)
    )
    score += 0.15 * structure_score
    
    # 5. Verbose explanations removed (15%)
    verbose_patterns = [
        'Portable Document Format', 'JavaScript Object Notation', 
        'Yet Another Markup Language', 'Application Programming Interface',
        'is an open-source', 'is a system for'
    ]
    verbose_count = sum(1 for p in verbose_patterns if p.lower() in optimized.lower())
    verbose_score = max(0, 1 - (verbose_count / 3))
    score += 0.15 * verbose_score
    
    return score

print("Defined skill_optimization_metric with code block preservation")
print("  - 25% Filler removal")
print("  - 20% Conciseness (40-70% target)")
print("  - 25% Code block preservation")
print("  - 15% Structure (frontmatter, sections)")
print("  - 15% Verbose removal")

Defined skill_optimization_metric with code block preservation
  - 25% Filler removal
  - 20% Conciseness (40-70% target)
  - 25% Code block preservation
  - 15% Structure (frontmatter, sections)
  - 15% Verbose removal


## 7. Run GEPA Optimization

In [13]:
# Run GEPA Optimization
# GEPA uses evolutionary strategies to optimize the prompt/instructions

from dspy import GEPA

# Create reflection LM (used for proposing new instructions)
# Using GPT-4o with higher temperature for creativity
reflection_lm = dspy.LM(
    "openai/gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=1.0,
    max_tokens=4096
)

# Initialize GEPA optimizer
gepa_optimizer = GEPA(
    metric=skill_optimization_metric,
    reflection_lm=reflection_lm,        # Required: LM for reflection/proposals
    max_full_evals=10,                  # Max full evaluations on trainset
    num_threads=1,                      # Parallel threads
    reflection_minibatch_size=3,        # Examples for reflection
    skip_perfect_score=True,            # Stop if perfect score achieved
)

print("Running GEPA optimization with GPT-4o...")
print("="*50)

# Compile/optimize the module
optimized_module = gepa_optimizer.compile(
    student=skill_optimizer_module,
    trainset=trainset,
)

print("GEPA optimization complete")


2026/04/08 14:01:37 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 10 metric calls of the program. This amounts to 10.00 full evals on the train set.
2026/04/08 14:01:37 WARNING dspy.teleprompt.gepa.gepa: No valset provided; Using trainset as valset. This is useful as an inference-time scaling strategy where you want GEPA to find the best solutions for the provided tasks in the trainset, as it makes GEPA overfit prompts to the provided trainset. In order to ensure generalization and perform well on unseen tasks, please provide separate trainset and valset. Provide the smallest valset that is just large enough to match the downstream task distribution, while keeping trainset as large as possible.
2026/04/08 14:01:37 INFO dspy.teleprompt.gepa.gepa: Using 1 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just l

Running GEPA optimization with GPT-4o...


GEPA Optimization:   0%|                           | 0/10 [00:00<?, ?rollouts/s]2026/04/08 14:01:47 INFO dspy.evaluate.evaluate: Average Metric: 0.8928571428571429 / 1 (89.3%)
2026/04/08 14:01:47 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.8928571428571429 over 1 / 1 examples
GEPA Optimization:  10%|█▉                 | 1/10 [00:10<01:35, 10.66s/rollouts]2026/04/08 14:01:47 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.8928571428571429


Average Metric: 2.68 / 3 (89.3%): 100%|███████████| 3/3 [00:00<00:00, 75.05it/s]

2026/04/08 14:01:47 INFO dspy.evaluate.evaluate: Average Metric: 2.678571428571429 / 3 (89.3%)


2026/04/08 14:01:57 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for optimizer.predict: Optimize a Kubernetes Helper Skill document by following these detailed steps:

1. **Preserve Essential Elements:**
   - Maintain all code blocks (e.g., ```bash, ```python), as these contain crucial commands necessary for Kubernetes operations.
   - Keep the frontmatter (--- name/description ---) to ensure proper metadata for the skill.
   - Retain section headers (##) for clear document structure.
   - Include useful technical content directly related to Kubernetes functionalities.

2. **Remove Redundant and Irrelevant Content:**
   - Eliminate filler phrases such as "make sure to," "ensure that," "don't forget to," etc.
   - Remove verbose explanations of basic concepts (YAML, JSON, API definitions) that are commonly understood by the target audience.
   - Exclude unrelated content, such as references to PDF processing tools, if they do not pertain to Kubernetes tasks.
   - Omit 

Average Metric: 2.68 / 3 (89.3%): 100%|██████████| 3/3 [00:00<00:00, 102.55it/s]

2026/04/08 14:02:11 INFO dspy.evaluate.evaluate: Average Metric: 2.678571428571429 / 3 (89.3%)
2026/04/08 14:02:11 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Proposed new text for optimizer.predict: Optimize a Kubernetes Helper Skill document by following these detailed steps:

1. **Preserve Essential Elements:**
   - Maintain all code blocks (e.g., ```bash, ```python), as these contain crucial commands necessary for Kubernetes operations.
   - Keep the frontmatter (--- name/description ---) to ensure proper metadata for the skill.
   - Retain section headers (##) for clear document structure.
   - Include useful technical content directly related to Kubernetes functionalities.

2. **Remove Redundant and Irrelevant Content:**
   - Eliminate filler phrases such as "make sure to," "ensure that," "don't forget to," etc.
   - Remove verbose explanations of basic concepts (YAML, JSON, API definitions) that are commonly understood by the target audience.
   - Exclude unrelated content, suc


GEPA optimization complete


## 8. Apply Optimized Module

In [14]:
# Use the GEPA-optimized module to optimize the skill
print("Optimizing skill with GEPA-trained module...")
print("="*50)

# Run the optimized module on the same content
gepa_result = optimized_module(
    original_content=skill_content,
    issues_found=issues_str
)

# Get the optimized content
optimized_content = gepa_result.optimized_content if hasattr(gepa_result, 'optimized_content') else str(gepa_result)

# Calculate metrics
original_len = len(skill_content)
optimized_len = len(optimized_content)
reduction = original_len - optimized_len
reduction_pct = (reduction / original_len) * 100 if original_len > 0 else 0

# Count code blocks
orig_blocks = count_code_blocks(skill_content)
opt_blocks = count_code_blocks(optimized_content)

print(f"\nResults:")
print(f"  Original: {original_len:,} chars, {orig_blocks} code blocks")
print(f"  Optimized: {optimized_len:,} chars, {opt_blocks} code blocks")
print(f"  Reduction: {reduction:,} chars ({reduction_pct:.1f}%)")
print(f"  Code blocks preserved: {opt_blocks}/{orig_blocks}")

# Check metric score
class PredictionWrapper:
    def __init__(self, content):
        self.optimized_content = content

metric_score = skill_optimization_metric(
    gold=trainset[0], 
    pred=PredictionWrapper(optimized_content),
    trace=None,
    pred_name=None,
    pred_trace=None
)
print(f"  Metric score: {metric_score:.2f}")

Optimizing skill with GEPA-trained module...

Results:
  Original: 8,000 chars, 14 code blocks
  Optimized: 2,575 chars, 8 code blocks
  Reduction: 5,425 chars (67.8%)
  Code blocks preserved: 8/14
  Metric score: 0.89


In [15]:
# Preview the optimized content
if 'optimized_content' in dir():
    print("GEPA-Optimized Skill Preview:")
    print("="*50)
    print(optimized_content[:2000])
    if len(optimized_content) > 2000:
        print("\n... (truncated)")

GEPA-Optimized Skill Preview:
---
name: Kubernetes_Helper_Skill_For_Managing_Cluster_Resources
description: Assist with managing resources, deployments, scaling, and troubleshooting in Kubernetes clusters.
---

# Kubernetes Helper Skill

## What This Skill Does

This skill assists with Kubernetes-related tasks, including managing cluster resources, deploying applications, scaling workloads, and troubleshooting issues.

## Prerequisites and Requirements

- **kubectl**: Ensure it is installed and correctly configured.
- **Kubernetes Cluster Access**: Confirm kubeconfig is properly set up and permissions are sufficient.

## How To Use This Skill

### Step 1: Connect to Your Cluster

Configure and verify kubectl context.

```bash
kubectl config get-contexts
kubectl config use-context <your-cluster>
kubectl cluster-info
kubectl get nodes
```

### Step 2: Check the Current State

Evaluate the existing state of cluster resources.

```bash
kubectl get pods [-n <namespace>] [--all-namespaces] [

In [16]:
# Save the optimized skill
if 'optimized_content' in dir():
    OUTPUT_DIR = Path("examples/Bad_Kubernetes_Helper_Skill_GEPA_Optimized_1")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    (OUTPUT_DIR / "SKILL.md").write_text(optimized_content)
    print(f"Saved GEPA-optimized skill to: {OUTPUT_DIR / 'SKILL.md'}")

Saved GEPA-optimized skill to: examples/Bad_Kubernetes_Helper_Skill_GEPA_Optimized_1/SKILL.md


## 9. Compare Results

In [17]:
# Compare original vs GEPA-optimized
if 'optimized_content' in dir() and skill:
    original = skill_content
    
    print("Comparison: Original vs GEPA-Optimized")
    print("="*60)
    print(f"{'Metric':<30} {'Original':<15} {'Optimized':<15}")
    print("-"*60)
    print(f"{'Characters':<30} {len(original):<15,} {len(optimized_content):<15,}")
    print(f"{'Lines':<30} {original.count(chr(10)):<15} {optimized_content.count(chr(10)):<15}")
    print(f"{'Est. Tokens':<30} {int(len(original)*0.25):<15,} {int(len(optimized_content)*0.25):<15,}")
    
    # Count code blocks
    orig_blocks = count_code_blocks(original)
    opt_blocks = count_code_blocks(optimized_content)
    print(f"{'Code Blocks':<30} {orig_blocks:<15} {opt_blocks:<15}")
    
    # Count filler phrases
    filler_phrases = ['make sure to', 'ensure that', "don't forget to", 'remember to', 'it is important to']
    orig_fillers = sum(original.lower().count(p) for p in filler_phrases)
    opt_fillers = sum(optimized_content.lower().count(p) for p in filler_phrases)
    print(f"{'Filler Phrases':<30} {orig_fillers:<15} {opt_fillers:<15}")
    
    reduction_pct = (1 - len(optimized_content)/len(original))*100
    print(f"\n{'Reduction':<30} {len(original) - len(optimized_content):,} chars ({reduction_pct:.1f}%)")
    print(f"{'Code Block Retention':<30} {opt_blocks}/{orig_blocks} ({100*opt_blocks/orig_blocks:.0f}%)" if orig_blocks > 0 else "")
else:
    print("No comparison available - run GEPA optimization first")

Comparison: Original vs GEPA-Optimized
Metric                         Original        Optimized      
------------------------------------------------------------
Characters                     8,000           2,575          
Lines                          194             96             
Est. Tokens                    2,000           643            
Code Blocks                    14              8              
Filler Phrases                 30              0              

Reduction                      5,425 chars (67.8%)
Code Block Retention           8/14 (57%)


## Summary

This notebook optimizes Claude Agent Skills using DSPy's **GEPA** optimizer.

### Key Improvements

The optimization now:
- **Preserves code blocks** (kubectl commands, scripts)
- **Consolidates** redundant command variations
- **Removes** filler phrases and verbose explanations
- **Penalizes** over-aggressive reduction

### Metric Weights

| Component | Weight | Description |
|-----------|--------|-------------|
| Filler removal | 25% | Remove "make sure to", "ensure that", etc. |
| Conciseness | 20% | Target 40-70% reduction (penalize over-reduction) |
| **Code preservation** | **25%** | Keep code blocks, consolidate variations |
| Structure | 15% | Preserve frontmatter, sections, code |
| Verbose removal | 15% | Remove acronym expansions |

### Expected Output

A well-optimized skill should:
- Remove all filler phrases
- Keep essential kubectl/bash commands
- Consolidate `kubectl get pods -n default/kube-system/my-namespace` into `kubectl get pods [-n <namespace>]`
- Remove unrelated content (PDF, Git commands)
- Achieve 40-70% size reduction while preserving functionality